In [15]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from criminisi import inpaint

def test_object_removal(image_path: str):
    # 1. Wczytaj obraz za pomocą OpenCV
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print(f"Błąd: Nie udało się wczytać obrazu z ścieżki: {image_path}")
        return
    
    # OpenCV wczytuje w formacie BGR, zamieniamy na standardowy RGB do wyświetlania
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape

    # 2. Utwórz maskę (True = obszar/obiekt do usunięcia, False = zostaje)
    # Na potrzeby testu definiujemy prostokąt w wybranym miejscu obrazu.
    # Możesz zmienić współrzędne, aby trafić w obiekt, który chcesz usunąć!
    mask = np.zeros((h, w), dtype=bool)
    
    y1, y2 = int(h*0.1), int(h * 0.9)  # np. pionowy środek
    x1, x2 = int(w*0.1), int(w * 0.9)  # np. poziomy środek
    mask[y1:y2, x1:x2] = True

    print("Trwa usuwanie obiektu i rekonstrukcja tła (algorytm Criminisiego)...")
    print("To może chwilę potrwać w zależności od wielkości zaznaczonego obszaru.")

    # 3. Uruchomienie algorytmu Criminisiego z naszego pliku criminisi.py
    # patch_size=9 oznacza wielkość łatki (puzzla) analizowanej przez algorytm
    result_image = inpaint(img_rgb, mask, patch_size=9)

    # 4. Wyświetlenie wyników przy użyciu Matplotlib
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img_rgb)
    axes[0].set_title("1. Obraz oryginalny")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("2. Maska (co usuwamy)")
    axes[1].axis("off")

    axes[2].imshow(result_image)
    axes[2].set_title("3. Wynik (po usunięciu)")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    # Podaj nazwę swojego pliku ze zdjęciem (np. "zdjecie.jpg")
    test_object_removal("pilka.png")

Trwa usuwanie obiektu i rekonstrukcja tła (algorytm Criminisiego)...
To może chwilę potrwać w zależności od wielkości zaznaczonego obszaru.


KeyboardInterrupt: 

In [16]:
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry, SamPredictor
from criminisi import inpaint

def remove_object_with_sam(image_path: str, checkpoint_path: str, input_point: np.ndarray, model_type: str = "vit_h"):
    # 1. Konfiguracja urządzenia (GPU lub CPU)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Uruchamianie modelu SAM na urządzeniu: {device}")

    # 2. Wczytanie obrazu za pomocą OpenCV
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print(f"Błąd: Nie udało się wczytać obrazu z ścieżki: {image_path}")
        return
    
    # Zamiana BGR na RGB do przetwarzania i wyświetlania
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w, _ = img_rgb.shape

    # 3. Inicjalizacja i uruchomienie modelu SAM w celu wygenerowania maski
    print("Trwa segmentacja obiektu za pomocą modelu SAM...")
    sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
    sam.to(device=device)
    predictor = SamPredictor(sam)
    
    predictor.set_image(img_rgb)
    
    # 1 oznacza punkt pierwszoplanowy (obiekt do usunięcia)
    input_label = np.array([1]) 
    
    masks, scores, logits = predictor.predict(
        point_coords=input_point,
        point_labels=input_label,
        multimask_output=True, # SAM generuje 3 warianty, wybierzemy najlepszy
    )

    # Wybór najlepszej maski na podstawie najwyższego wyniku (score)
    best_mask_idx = np.argmax(scores)
    mask = masks[best_mask_idx] # Maska typu boolean (True/False)

    print("Maska gotowa. Trwa usuwanie obiektu i rekonstrukcja tła (algorytm Criminisiego)...")
    print("To może chwilę potrwać w zależności od wielkości wyznaczonego obszaru.")

    # 4. Uruchomienie algorytmu Criminisiego z wygenerowaną przez SAM maską
    result_image = inpaint(img_rgb, mask, patch_size=9)

    # 5. Wyświetlenie wyników przy użyciu Matplotlib
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img_rgb)
    axes[0].set_title("1. Obraz oryginalny")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("2. Maska z SAM (co usuwamy)")
    axes[1].axis("off")

    axes[2].imshow(result_image)
    axes[2].set_title("3. Wynik (po usunięciu)")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    # Konfiguracja ścieżek i punktu na obiekcie
    IMAGE_PATH = "samolot.jpg"          # Ścieżka do Twojego zdjęcia
    CHECKPOINT_PATH = r"C:\Users\OliLa\sam_vit_h_4b8939.pth" # Ścieżka do wag modelu SAM
    
    # Współrzędne [x, y] punktu znajdującego się na obiekcie, który chcesz usunąć
    # (np. środek obiektu w pikselach)
    point = np.array([[373, 606]]) 

    remove_object_with_sam(
        image_path=IMAGE_PATH,
        checkpoint_path=CHECKPOINT_PATH,
        input_point=point,
        model_type="vit_h"
    )

Uruchamianie modelu SAM na urządzeniu: cpu
Trwa segmentacja obiektu za pomocą modelu SAM...
Maska gotowa. Trwa usuwanie obiektu i rekonstrukcja tła (algorytm Criminisiego)...
To może chwilę potrwać w zależności od wielkości wyznaczonego obszaru.


KeyboardInterrupt: 

In [9]:
import cv2

def click_event(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        print(f"--> Wyklikany punkt dla SAM: np. np.array([[{x}, {y}]])")
        # Wyświetli współrzędne bezpośrednio w konsoli

# Wczytaj swój obraz
image_path = "samolot.jpg"
img = cv2.imread(image_path)

if img is None:
    print(f"Nie znaleziono pliku: {image_path}")
else:
    cv2.imshow("Kliknij w środek pilki", img)
    cv2.setMouseCallback("Kliknij w środek pilki", click_event)
    print("Kliknij w dowolne miejsce na piłce w otwartym oknie obrazu...")
    cv2.waitKey(0)
    cv2.destroyAllWindows()

Kliknij w dowolne miejsce na piłce w otwartym oknie obrazu...
--> Wyklikany punkt dla SAM: np. np.array([[368, 620]])
--> Wyklikany punkt dla SAM: np. np.array([[373, 606]])
